In [8]:
import json
import re
import numpy as np
import torch
import torch.nn as nn
import joblib
import pandapower as pp
import pandapower.networks as pn
import pandas as pd
import random
import ipywidgets as widgets
from IPython.display import display, clear_output
import os

# Robustly find and move to the project root, regardless of where the kernel started
target_folder = "grid-congestion-project"
current = os.getcwd()

if os.path.basename(current) != target_folder:
    # Try the known direct path first
    known_path = r"C:\Users\HP\Music\grid-congestion-project"
    if os.path.isdir(known_path):
        os.chdir(known_path)
    else:
        # Fallback: walk upward looking for a folder containing 'app' and 'experiments'
        probe = current
        for _ in range(5):
            if os.path.isdir(os.path.join(probe, "app")) and os.path.isdir(os.path.join(probe, "experiments")):
                os.chdir(probe)
                break
            probe = os.path.dirname(probe)

print("Working dir:", os.getcwd())
print("Imports OK")

Working dir: C:\Users\HP\Music\grid-congestion-project
Imports OK


In [7]:
def load_models():
    model1 = joblib.load("app/model1_baseline.pkl")
    with open("app/vocab.json", "r") as f:
        vocab = json.load(f)

    class MultiModalCongestionModel(nn.Module):
        def __init__(self, vocab_size, struct_dim=9, embed_dim=16, hidden_dim=32):
            super().__init__()
            self.struct_branch = nn.Sequential(
                nn.Linear(struct_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, hidden_dim)
            )
            self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
            self.text_lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
            self.fusion = nn.Sequential(
                nn.Linear(hidden_dim * 2, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, 1)
            )
        def forward(self, structured, text_seq):
            struct_out = self.struct_branch(structured)
            embedded = self.embedding(text_seq)
            _, (hidden, _) = self.text_lstm(embedded)
            text_out = hidden.squeeze(0)
            combined = torch.cat([struct_out, text_out], dim=1)
            return self.fusion(combined).squeeze(1)

    model2 = MultiModalCongestionModel(vocab_size=len(vocab))
    model2.load_state_dict(torch.load("app/model2_multimodal.pt", map_location="cpu"))
    model2.eval()
    return model1, model2, vocab

def load_test_scenarios():
    data = []
    with open("data/processed/test_multimodal.jsonl", "r") as f:
        for line in f:
            data.append(json.loads(line))
    return data

model1, model2, vocab = load_models()
test_scenarios = load_test_scenarios()
print(f"Loaded model1, model2, vocab ({len(vocab)} tokens), and {len(test_scenarios)} test scenarios")

Loaded model1, model2, vocab (98 tokens), and 319 test scenarios


In [6]:
def tokenize(text):
    return re.findall(r"[a-z]+", text.lower())

def text_to_sequence(text, vocab, max_len=30):
    tokens = tokenize(text)
    seq = [vocab.get(tok, vocab["<UNK>"]) for tok in tokens]
    seq = seq[:max_len]
    seq = seq + [vocab["<PAD>"]] * (max_len - len(seq))
    return seq

def extract_features_model1(wind_speed, solar, storm, load_mult, total_wind, total_solar, has_outage, outage_storm, outage_equip):
    return [wind_speed, solar, int(storm), load_mult, total_wind, total_solar, has_outage, outage_storm, outage_equip]

def extract_features_model2(wind_speed, solar, storm, load_mult, total_wind, total_solar, has_outage, outage_storm, outage_equip):
    return [
        wind_speed / 90.0, solar / 1000.0, int(storm), (load_mult - 0.8) / 1.4,
        total_wind / 240.0, total_solar / 160.0, has_outage, outage_storm, outage_equip
    ]

def run_redispatch(wind_speed, solar_irr, load_mult, has_outage, outage_line_id):
    net = pn.case118()

    # FIX 1: explicit voltage bounds — needed for OPF to converge reliably
    net.bus['min_vm_pu'] = 0.90
    net.bus['max_vm_pu'] = 1.10

    # FIX 2: relaxed thermal derating (was 0.07, too aggressive)
    net.line['max_i_ka'] = net.line['max_i_ka'] * 0.15

    gen_buses = net.gen['bus'].tolist()
    random.seed(42)
    renewable_buses = random.sample(gen_buses, min(10, len(gen_buses)))

    for bus in renewable_buses:
        wind_mw = max(0, (wind_speed / 90) * 80)
        solar_mw = max(0, (solar_irr / 1000) * 50)
        total_mw = wind_mw + solar_mw
        max_cap = max(total_mw * 4.0, 50)
        pp.create_sgen(net, bus=bus, p_mw=total_mw, name=f"renewable_bus{bus}", controllable=True,
                        min_p_mw=0, max_p_mw=max_cap, min_q_mvar=-max_cap*0.3, max_q_mvar=max_cap*0.3)

    net.load['p_mw'] = net.load['p_mw'] * load_mult
    net.load['q_mvar'] = net.load['q_mvar'] * load_mult

    if has_outage and outage_line_id is not None and outage_line_id in net.line.index:
        net.line.at[outage_line_id, 'in_service'] = False

    try:
        pp.runpp(net, enforce_q_lims=False, max_iteration=15, numba=False, init="flat")
        max_loading_before = net.res_line['loading_percent'].max()
        conv_before = net.gen['p_mw'].sum()
        renew_before = net.sgen['p_mw'].sum()
        renew_pct_before = (renew_before / (conv_before + renew_before) * 100) if (conv_before+renew_before) > 0 else 0
    except Exception:
        return None

    net.poly_cost = net.poly_cost.iloc[0:0]
    for idx in net.gen.index:
        net.gen.at[idx, 'controllable'] = True
        net.gen.at[idx, 'min_p_mw'] = 0
        net.gen.at[idx, 'max_p_mw'] = max(net.gen.at[idx, 'max_p_mw'], net.gen.at[idx, 'p_mw'] * 1.2 + 1)
        if pd.isna(net.gen.at[idx, 'min_q_mvar']):
            net.gen.at[idx, 'min_q_mvar'] = -net.gen.at[idx, 'max_p_mw'] * 0.5
        if pd.isna(net.gen.at[idx, 'max_q_mvar']):
            net.gen.at[idx, 'max_q_mvar'] = net.gen.at[idx, 'max_p_mw'] * 0.5
        pp.create_poly_cost(net, idx, 'gen', cp1_eur_per_mw=100)
    for idx in net.sgen.index:
        pp.create_poly_cost(net, idx, 'sgen', cp1_eur_per_mw=5)

    try:
        pp.runopp(net, numba=False)
        max_loading_after = net.res_line['loading_percent'].max()
        resolved = bool(max_loading_after <= 100)
        conv_after = net.res_gen['p_mw'].sum()
        renew_after = net.res_sgen['p_mw'].sum()
        renew_pct_after = (renew_after / (conv_after + renew_after) * 100) if (conv_after+renew_after) > 0 else 0
        return {
            "converged": True, "resolved": resolved,
            "loading_before": round(max_loading_before, 2), "loading_after": round(max_loading_after, 2),
            "renewable_before": round(renew_pct_before, 1), "renewable_after": round(renew_pct_after, 1),
        }
    except Exception:
        return {"converged": False, "loading_before": round(max_loading_before, 2)}

print("Feature extraction and redispatch functions defined (v2: voltage bounds + relaxed thermal limit)")

Feature extraction and redispatch functions defined (v2: voltage bounds + relaxed thermal limit)


In [ ]:
scenario_ids = [s["scenario_id"] for s in test_scenarios]

mode_dropdown = widgets.Dropdown(
    options=["Pick from test scenarios", "Custom scenario (sliders)"],
    value="Pick from test scenarios",
    description="Input mode:",
    style={'description_width': 'initial'}
)

scenario_dropdown = widgets.Dropdown(
    options=scenario_ids,
    description="Test scenario:",
    style={'description_width': 'initial'}
)

wind_slider = widgets.FloatSlider(value=45.0, min=0.0, max=90.0, step=1.0, description="Wind speed (km/h):", style={'description_width': 'initial'})
solar_slider = widgets.FloatSlider(value=500.0, min=0.0, max=1000.0, step=10.0, description="Solar irradiance (W/m²):", style={'description_width': 'initial'})
load_slider = widgets.FloatSlider(value=1.2, min=0.8, max=2.2, step=0.05, description="Load multiplier:", style={'description_width': 'initial'})
outage_checkbox = widgets.Checkbox(value=False, description="Line outage event")

custom_box = widgets.VBox([wind_slider, solar_slider, load_slider, outage_checkbox])
custom_box.layout.display = 'none'

run_button = widgets.Button(description="🔍 Run Prediction & Redispatch", button_style='primary', layout=widgets.Layout(width='300px'))
output = widgets.Output()

def on_mode_change(change):
    if change['new'] == "Custom scenario (sliders)":
        scenario_dropdown.layout.display = 'none'
        custom_box.layout.display = 'block'
    else:
        scenario_dropdown.layout.display = 'block'
        custom_box.layout.display = 'none'

mode_dropdown.observe(on_mode_change, names='value')

def get_inputs():
    if mode_dropdown.value == "Pick from test scenarios":
        scenario = next(s for s in test_scenarios if s["scenario_id"] == scenario_dropdown.value)
        wind_speed = scenario["weather"]["wind_speed_kmh"]
        solar_irr = scenario["weather"]["solar_irradiance_wm2"]
        storm = scenario["weather"]["storm_flag"]
        load_mult = scenario["load_multiplier"]
        total_wind = sum(v["wind_mw"] for v in scenario["renewable_output"].values())
        total_solar = sum(v["solar_mw"] for v in scenario["renewable_output"].values())
        has_outage = 1 if scenario["event"]["type"] == "line_outage" else 0
        outage_storm = 1 if scenario["event"].get("cause") == "storm" else 0
        outage_equip = 1 if scenario["event"].get("cause") == "equipment_failure" else 0
        outage_line_id = scenario["event"].get("line_id")
        text_report = scenario["text_report"]
        actual_label = scenario["congested"]
        return wind_speed, solar_irr, storm, load_mult, total_wind, total_solar, has_outage, outage_storm, outage_equip, outage_line_id, text_report, actual_label
    else:
        wind_speed = wind_slider.value
        solar_irr = solar_slider.value
        load_mult = load_slider.value
        storm = wind_speed > 65
        has_outage = 1 if outage_checkbox.value else 0
        outage_storm = 1 if (has_outage and storm) else 0
        outage_equip = 1 if (has_outage and not storm) else 0
        outage_line_id = 23 if has_outage else None
        total_wind = (wind_speed / 90) * 80 * 10
        total_solar = (solar_irr / 1000) * 50 * 10
        text_report = f"Custom scenario: wind {wind_speed:.0f} km/h, load {load_mult:.2f}x, outage={bool(has_outage)}"
        actual_label = None
        return wind_speed, solar_irr, storm, load_mult, total_wind, total_solar, has_outage, outage_storm, outage_equip, outage_line_id, text_report, actual_label

def on_run_clicked(b):
    with output:
        clear_output()
        wind_speed, solar_irr, storm, load_mult, total_wind, total_solar, has_outage, outage_storm, outage_equip, outage_line_id, text_report, actual_label = get_inputs()

        print(f"📄 Scenario text: {text_report}")
        if actual_label is not None:
            print(f"   Actual label: {'CONGESTED' if actual_label else 'Not congested'}")
        print()

        feat1 = extract_features_model1(wind_speed, solar_irr, storm, load_mult, total_wind, total_solar, has_outage, outage_storm, outage_equip)
        pred1 = model1.predict([feat1])[0]
        prob1 = model1.predict_proba([feat1])[0][1]

        feat2 = extract_features_model2(wind_speed, solar_irr, storm, load_mult, total_wind, total_solar, has_outage, outage_storm, outage_equip)
        text_seq = text_to_sequence(text_report, vocab)
        with torch.no_grad():
            out = model2(torch.tensor([feat2], dtype=torch.float32), torch.tensor([text_seq], dtype=torch.long))
            prob2 = torch.sigmoid(out).item()
            pred2 = 1 if prob2 > 0.5 else 0

        print("── Predictions ──")
        print(f"Model 1 (Structured Only):    {'CONGESTED' if pred1 else 'Not Congested'}  ({prob1*100:.1f}% confidence)")
        print(f"Model 2 (Structured + Text):  {'CONGESTED' if pred2 else 'Not Congested'}  ({prob2*100:.1f}% confidence)")
        print()

        if pred1 or pred2:
            print("── Redispatch Optimization ──")
            print("Running OPF optimization...")
            result = run_redispatch(wind_speed, solar_irr, load_mult, has_outage, outage_line_id)

            if result is None:
                print("❌ Base power flow did not converge for this scenario.")
            elif not result["converged"]:
                print(f"⚠️ OPF could not find a feasible solution. Max loading before: {result['loading_before']}%")
            else:
                print(f"Line Loading:      {result['loading_before']}% → {result['loading_after']}%  ({result['loading_after']-result['loading_before']:+.1f}%)")
                print(f"Renewable Usage:   {result['renewable_before']}% → {result['renewable_after']}%  ({result['renewable_after']-result['renewable_before']:+.1f}%)")
                print(f"Congestion Status: {'✅ RESOLVED' if result['resolved'] else '❌ NOT RESOLVED'}")
        else:
            print("✅ No congestion predicted — redispatch not needed.")

run_button.on_click(on_run_clicked)

display(mode_dropdown, scenario_dropdown, custom_box, run_button, output)

Dropdown(description='Input mode:', options=('Pick from test scenarios', 'Custom scenario (sliders)'), style=D…

Dropdown(description='Test scenario:', options=('SC_00393', 'SC_01342', 'SC_01680', 'SC_02300', 'SC_00800', 'S…

Button(button_style='primary', description='🔍 Run Prediction & Redispatch', layout=Layout(width='300px'), styl…

Output()

In [ ]:
import os
print("Current working dir:", os.getcwd())
print("Does app/ exist here?", os.path.isdir("app"))
print("Does app/model1_baseline.pkl exist?", os.path.isfile("app/model1_baseline.pkl"))